# 10 — Global admin0 multi-hazard overlap (children)

How many boys and girls are exposed **simultaneously** to several climate hazards, e.g. *girls exposed to at least 3*. Overlap needs per-pixel data, so unlike notebook 09 this runs in Earth Engine; see `pipeline/multi_hazard.py`.

**Hazards (8 topics):** River Flood, Coastal Flood, Tropical Storm, Drought, Heatwave, Extreme Heat, Fire, Sand & Dust Storm. A topic applies to a pixel if any of its layers exceeds its threshold.

## Flow

1. **Pilot** (interactive, small countries, no tasks): writes the same CSV shape as an export.
2. **Submit**: one `Export.table.toDrive` task per country. Only runs with `RUN_GLOBAL = True`. Resumable through the manifest.
3. Download the Drive folder into `EXPORT_DIR`.
4. **Assemble + verify**. Pure pandas; no further GEE needed.

## Outputs

- `{OUTPUT_FOLDER}/global_admin0/multi_hazard_children_at_least_{date}.xlsx`
- `{OUTPUT_FOLDER}/global_admin0/multi_hazard_children_exactly_{date}.xlsx`
- `{OUTPUT_FOLDER}/global_admin0/CSV/multi_hazard_combinations_{date}.csv` — every combination, country × code

## Model

- Hazard thresholds from `hazard_info.json`; population from the WorldPop mosaics on their native grid; geometry from the 500 km adm0 chunks.
- A pixel outside a hazard raster's coverage counts as not exposed.
- Hazards with Insufficient Data for a country are cleared, so its "at least N" covers fewer hazards: see `hazards_assessed`.
- `agg.POP_BLANK_COUNTRIES` get no row.

In [ ]:
import sys
from pathlib import Path

SUBNATIONAL_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline" / "settings.py").is_file())
if str(SUBNATIONAL_ROOT) not in sys.path:
    sys.path.insert(0, str(SUBNATIONAL_ROOT))
from pipeline.settings import Settings
OUTPUT_ROOT = Settings.from_env().storage_root   # notebooks/output unless GCHD_STORAGE_ROOT is set
LIB_DIR = SUBNATIONAL_ROOT / "lib"
CONFIG_DIR = SUBNATIONAL_ROOT / "config"
for p in (str(SUBNATIONAL_ROOT), str(LIB_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import json
import os
import pandas as pd
import aggregation_functions as agg
from pipeline import auth, excel_report as xr, multi_hazard as mh
from pipeline.hazards import load_hazards
from pipeline.settings import Settings
from pipeline.steps.s00 import _await_admission

pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters
# ============================================================
RUN_GLOBAL = True   # False = pilot only. True submits ~240 EE tasks.
PILOT = ["ISL_V1", "LIE_V1", "FJI_V2", "SRB_V1"]

PROD_DATE = "2026-08-31"
SOURCE_FOLDER = f"{OUTPUT_ROOT}/global/CSV"
OUTPUT_FOLDER = OUTPUT_ROOT
POP_REFERENCE = f"{OUTPUT_FOLDER}/aux/admin2_population.csv"
NB09_LONG = f"{OUTPUT_FOLDER}/global_admin0/CSV/children_exposure_long_{PROD_DATE}.csv"

DRIVE_FOLDER = "gchd_multi_hazard_exports"
MANIFEST = f"{OUTPUT_FOLDER}/aux/multi_hazard_manifest_{PROD_DATE}.csv"
# Download the Drive folder here after the tasks finish.
EXPORT_DIR = f"{OUTPUT_FOLDER}/aux/multi_hazard_exports_{PROD_DATE}"
PILOT_DIR = f"{OUTPUT_FOLDER}/aux/multi_hazard_pilot_{PROD_DATE}"

SUFFIX = "" if RUN_GLOBAL else "_pilot"
for d in (EXPORT_DIR, PILOT_DIR, f"{OUTPUT_FOLDER}/global_admin0/CSV"):
    os.makedirs(d, exist_ok=True)

# Country list and names: same rows as notebook 09.
any_code = agg.discover_hazard_codes(SOURCE_FOLDER, PROD_DATE)[0]
src = agg.load_hazard_global(any_code, PROD_DATE, SOURCE_FOLDER)
src = src[~src["adm0_ucode"].isin(agg.POP_BLANK_COUNTRIES)]
countries = (src[["iso3", "adm0_ucode", "adm0_name", "unicef_region"]]
             .drop_duplicates("adm0_ucode").set_index("adm0_ucode"))
print(f"countries: {len(countries)}")

settings = Settings.from_env()
ee, utils = auth.ee_and_utils(settings)
stack, grid = mh.build_stack(ee, utils, load_hazards(settings))
print("grid:", grid)

In [ ]:
# ============================================================
# Pilot: small countries interactively, no tasks submitted
# ============================================================
if not RUN_GLOBAL:
    for u in PILOT:
        fc = mh.reduce_country(ee, stack, grid, u, settings.tile_scale).getInfo()
        rows = [{"ucode": u, "groups": json.dumps(f["properties"].get("groups", []))}
                for f in fc["features"]]
        pd.DataFrame(rows).to_csv(f"{PILOT_DIR}/{u}.csv", index=False)
        print(f"  {u}: {len(rows)} chunks")

In [ ]:
# ============================================================
# Submit one export per country (RUN_GLOBAL only). Resumable.
# ============================================================
if RUN_GLOBAL:
    done = pd.read_csv(MANIFEST) if os.path.exists(MANIFEST) else pd.DataFrame(columns=["ucode", "task_id"])
    todo = [u for u in countries.index if u not in set(done["ucode"])]
    print(f"submitting {len(todo)} of {len(countries)}")
    submitted = []
    try:
        for u in todo:
            _await_admission(ee, settings)
            task = ee.batch.Export.table.toDrive(
                collection=mh.reduce_country(ee, stack, grid, u, settings.tile_scale),
                description=f"mhz_{u}"[:100], folder=DRIVE_FOLDER,
                fileNamePrefix=u, fileFormat="CSV", selectors=["ucode", "groups"],
            )
            task.start()
            submitted.append({"ucode": u, "task_id": task.id})
    finally:
        pd.concat([done, pd.DataFrame(submitted)]).to_csv(MANIFEST, index=False)
    print(f"submitted {len(submitted)} -> {MANIFEST}")

In [ ]:
# ============================================================
# Assemble: combinations, suppression, at-least / exactly
# ============================================================
raw = mh.load_exports(EXPORT_DIR if RUN_GLOBAL else PILOT_DIR)
raw = raw[raw["adm0_ucode"].isin(countries.index)]
long = mh.apply_suppression(raw)

combos = long.assign(n_hazards=mh.n_hazards(long["code"]),
                     hazards=long["code"].map(mh.topic_names))
combos_path = f"{OUTPUT_FOLDER}/global_admin0/CSV/multi_hazard_combinations_{PROD_DATE}{SUFFIX}.csv"
combos.round(0).to_csv(combos_path, index=False)
print(f"combinations: {combos.shape} -> {combos_path}")

front = countries.loc[sorted(long["adm0_ucode"].unique())].reset_index()
masks = front["adm0_ucode"].map(mh.suppressed_bits)
front["hazards_assessed"] = mh.N_TOPICS - mh.n_hazards(masks)
front["hazards_suppressed"] = masks.map(lambda m: "" if m == 0 else mh.topic_names(m))

POP_GROUP = "child_population"
LABEL = {"pop_t": "Total", "pop_m": "Boys", "pop_f": "Girls"}
FRONT = ["iso3", "adm0_ucode", "adm0_name", "unicef_region", "hazards_assessed", "hazards_suppressed"]
FRONT_LABELS = {"iso3": "ISO3", "adm0_ucode": "Country Code", "adm0_name": "Country",
                "unicef_region": "UNICEF Region", "hazards_assessed": "Hazards Assessed",
                "hazards_suppressed": "Hazards Suppressed"}
cfg = xr.load_report_config(CONFIG_DIR)

def write_view(table, title, ns, stem):
    """One sheet: Child Population, then a Total | Boys | Girls group per n."""
    wide = front.set_index("adm0_ucode", drop=False)
    pop = mh.at_least_table(long).query("n == 0").set_index("adm0_ucode")
    for c in mh.SUM_COLS:
        wide[f"{POP_GROUP}__{c}"] = pop[c].round(0)
    display = {POP_GROUP: "Child Population (under 18)"}
    spans = [(POP_GROUP, [f"{POP_GROUP}__{c}" for c in mh.SUM_COLS])]
    for n in ns:
        key = f"{stem}_{n}"
        sub = table[table["n"] == n].set_index("adm0_ucode")
        for c in mh.SUM_COLS:
            wide[f"{key}__{c}"] = sub[c].round(0)
        display[key] = title.format(n=n)
        spans.append((key, [f"{key}__{c}" for c in mh.SUM_COLS]))
    path = f"{OUTPUT_FOLDER}/global_admin0/multi_hazard_children_{stem}_{PROD_DATE}{SUFFIX}.xlsx"
    print(xr.write_wide_excel(
        wide.reset_index(drop=True), path,
        cfg={**cfg, "code_display": {**cfg["code_display"], **display}},
        front_cols=FRONT, group_spans=spans, class_value_suffixes=set(),
        front_labels=FRONT_LABELS, variable_labels=LABEL,
        thousands_cols={c for _, cols in spans for c in cols}, sheet_name=stem,
    ))

at_least = mh.at_least_table(long)
exactly = mh.exactly_table(long)
write_view(at_least, "At least {n}", range(1, mh.N_TOPICS + 1), "at_least")
write_view(exactly, "Exactly {n}", range(0, mh.N_TOPICS + 1), "exactly")

In [ ]:
# ============================================================
# Verification
# ============================================================
pop = at_least.query("n == 0").set_index("adm0_ucode")

# 1. Population vs the admin2 reference.
ref = (pd.read_csv(POP_REFERENCE).merge(src[["adm2_ucode", "adm0_ucode"]], on="adm2_ucode")
       .groupby("adm0_ucode")["pop_under_18_total"].sum())
gap = (pop["pop_t"] / ref.reindex(pop.index) - 1) * 100
if RUN_GLOBAL:
    print(f"countries missing from the results: {sorted(set(countries.index) - set(pop.index))}")
print(f"population vs reference: largest gap {gap.abs().max():.3f}%")
print("  over 0.5%:", gap[gap.abs() > 0.5].round(2).to_dict())

# 2. Marginals vs notebook 09 (which ceils per admin2, so sits slightly above).
nb10 = pd.read_csv(NB09_LONG)
marg = mh.topic_marginals(raw)
bad = []
for topic, layers in mh.TOPICS:
    sub = nb10[nb10["hazard_code"].isin(layers)].pivot(index="adm0_ucode", columns="hazard_code", values="u18_aexp")
    m = marg[marg["topic"] == topic].set_index("adm0_ucode")["pop_t"].reindex(pop.index).fillna(0)
    low = sub.reindex(pop.index).max(axis=1).fillna(0)
    high = sub.reindex(pop.index).sum(axis=1)
    # Slack covers the per-admin2 ceiling in the source tables.
    slack = 0.01 * pop["pop_t"].clip(upper=1e5) + 25
    off = pop.index[(m < low - slack) | (m > high + slack)]
    bad += [(topic, u, round(m[u]), round(low[u]), round(high[u])) for u in off]
print("topic marginals outside [largest layer, sum of layers]:", bad or "none")

# 3. Internal consistency.
print("at-least never rises with n:", (at_least.groupby("adm0_ucode")["pop_t"].diff().dropna() <= 1e-6).all())
print("exactly sums to population:", ((exactly.groupby("adm0_ucode")["pop_t"].sum() - pop["pop_t"]).abs() < 1).all())
d = (pop["pop_m"] + pop["pop_f"] - pop["pop_t"]).abs() / pop["pop_t"].where(pop["pop_t"] > 0)
print(f"boys + girls vs total: largest gap {d.max():.3%}")

# 4. The headline.
g = at_least.query("n == 3")
print(f"\nchildren exposed to at least 3 hazards: {g['pop_t'].sum():,.0f} "
      f"(boys {g['pop_m'].sum():,.0f}, girls {g['pop_f'].sum():,.0f})")